In [5]:
import mlflow
import pandas as pd
import mlflow.sklearn

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import string
import pandas as pd
import numpy as np
import re


In [6]:
df = pd.read_csv('IMDB.csv')
df.head()

,review,sentiment
0,Film version of Sandra Bernhard's one-woman of...,negative
1,I switched this on (from cable) on a whim and ...,positive
2,The `plot' of this film contains a few holes y...,negative
3,"Some amusing humor, some that falls flat, some...",negative
4,What can you say about this movie? It was not ...,negative


In [7]:
df.shape

(1000, 2)

In [8]:
## data preprocessing

# Define text preprocessing fucntions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatize = WordNetLemmatizer()
    text = text.split()
    text = [lemmatize.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Removing numbers from the text."""
    text = "".join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), " ", text)
    text = text.replace("؛", "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
/var/folders/n8/j0_rrlzd54b4xk3yllh67k000000gn/T/ipykernel_88842/3736354925.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
0,film version sandra bernhard s one woman off b...,negative
1,switched from cable whim treated quite surpris...,positive
2,plot film contains hole could drive massive tr...,negative
3,amusing humor fall flat decent acting quite at...,negative
4,say movie terrible good two day earlier watche...,negative


In [10]:
df.sentiment.value_counts()

sentiment
negative    517
positive    483
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
0,film version sandra bernhard s one woman off b...,0
1,switched from cable whim treated quite surpris...,1
2,plot film contains hole could drive massive tr...,0
3,amusing humor fall flat decent acting quite at...,0
4,say movie terrible good two day earlier watche...,0


In [12]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [13]:
vectorize = CountVectorizer(max_features=100)
X = vectorize.fit_transform(df['review'])
y = df['sentiment']

array(<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 22 stored elements and shape (1, 100)>, dtype=object)

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [31]:
import dagshub

dagshub.init(repo_owner='ashishpal003', repo_name='Movie-Review-Sentiment-Prediction', mlflow=True)

2026-03-07 16:29:27,620 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/ashishpal003/Movie-Review-Sentiment-Prediction "HTTP/1.1 200 OK"


Initialized MLflow to track repo "ashishpal003/Movie-Review-Sentiment-Prediction"

2026-03-07 16:29:27,625 - INFO - Initialized MLflow to track repo "ashishpal003/Movie-Review-Sentiment-Prediction"


Repository ashishpal003/Movie-Review-Sentiment-Prediction initialized!

2026-03-07 16:29:27,626 - INFO - Repository ashishpal003/Movie-Review-Sentiment-Prediction initialized!


In [38]:
# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic-Regression Baseline")

2026/03/07 16:36:32 INFO mlflow.tracking.fluent: Experiment with name 'Logistic-Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ee4e719e7410474b9167215ed3258ff5', creation_time=1772881593160, experiment_id='1', last_update_time=1772881593160, lifecycle_stage='active', name='Logistic-Regression Baseline', tags={}, workspace='default'>

In [39]:
import mlflow
import logging
import os
import time

# Configure the logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

## check if dagshub has set the tracking uri
if mlflow.get_tracking_uri():
    logging.info(f"Tracking uri is set at: {mlflow.get_tracking_uri()}")
else:
    raise "Error: Tracking uri Not found"

logging.info("Starting MLFlow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000) # maximum number of iterations for solver(convergence function) to converge

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-03-07 16:36:42,874 - INFO - Tracking uri is set at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow
2026-03-07 16:36:42,875 - INFO - Starting MLFlow run...
2026-03-07 16:36:43,417 - INFO - Logging preprocessing parameters...
2026-03-07 16:36:44,399 - INFO - Initializing Logistic Regression model...
2026-03-07 16:36:44,400 - INFO - Fitting the model...
2026-03-07 16:36:44,411 - INFO - Model training complete.
2026-03-07 16:36:44,411 - INFO - Logging model parameters...
2026-03-07 16:36:44,804 - INFO - Making predictions...
2026-03-07 16:36:44,805 - INFO - Calculating evaluation metrics...
2026-03-07 16:36:44,813 - INFO - Logging evaluation metrics...
2026-03-07 16:36:46,026 - INFO - Saving and logging the model...
2026/03/07 16:36:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 16:36:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution beca

🏃 View run agreeable-ray-3 at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/1/runs/5185381745664ce8961f079341014061
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/1
